# CS439 Homework: Naive Bayes Spam Detection (Spring 2026)

In this homework you will build a **Naive Bayes** text classifier from scratch to detect spam SMS messages.

You will:
- load and explore the dataset,
- preprocess text (tokenization, normalization, stop-word removal),
- estimate Naive Bayes probabilities with **Laplace smoothing**,
- make predictions with log-probabilities,
- evaluate accuracy / precision / recall / F1, and
- write a short discussion.

**Dataset:** We use the *SMS Spam Collection* dataset. The file `SMSSpamCollection` is expected to be in the same folder as this notebook. Each line has the format:

```
<label>	<message>
```

where `<label>` is `ham` or `spam`.


## Instructions

- Fill in code where you see `TODO`.
- Keep your code **readable** and **well-tested**.
- Before submitting: **restart kernel** and **Run All**.

We use placeholders like:

```python
# TODO
raise NotImplementedError
```

Delete the `raise NotImplementedError` once you finish that part.

<div class="alert alert-block alert-info">
<h3>Student Information</h3>
<b>Name</b>: Raj Shah<br>
<b>NetID</b>: ras637<br>
<b>Recitation #</b>: 01:198:439:05<br>
<b>Notes to Grader</b> (optional):<br>

<b>IMPORTANT</b>
Your work will not be graded withour your initials below<br>
I certify that this lab represents my own work and I have read the RU academic intergrity policies at<br>
<a href="https://www.cs.rutgers.edu/academics/undergraduate/academic-integrity-policy">https://www.cs.rutgers.edu/academics/undergraduate/academic-integrity-policy </a><br>

<b>Initials</b>: RS 
</div>


## Question 1: Load and explore the dataset

Tasks:
1. Load `SMSSpamCollection` into a list of `(label, message)` pairs.
2. Print the total number of messages.
3. Print the first 5 examples.
4. Print the number of `ham` vs `spam` messages.

**Hint:** `label, message = line.split("	", 1)`.


In [1]:
from pathlib import Path
from collections import Counter

def load_sms_spam_collection(path: Path) -> list[tuple[str, str]]:
    """Load the dataset

    TODO: Implement.
    """
    data = []
    
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            label, message = line.split("\t", 1)
            data.append((label, message))
    return data

DATA_PATH = Path('SMSSpamCollection.txt') # your data path
data = load_sms_spam_collection(DATA_PATH)
print('Total messages:')
print(len(data))
# TODO: Print the total number of messages

print('First 5 examples:')
for example in data[:5]: 
    print(example)
# TODO: Print the first 5 examples

print('Label counts:')
label_counts = Counter(label for label, message in data)
print(label_counts)
# TODO: Print the number of `ham` vs `spam` messages.

Total messages:
5574
First 5 examples:
('ham', 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...')
('ham', 'Ok lar... Joking wif u oni...')
('spam', "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's")
('ham', 'U dun say so early hor... U c already then say...')
('ham', "Nah I don't think he goes to usf, he lives around here though")
Label counts:
Counter({'ham': 4827, 'spam': 747})


## Question 2: Text preprocessing (tokenization + normalization)

Implement `preprocess_text(text)` that:
- lowercases,
- replaces non-alphanumeric characters with spaces,
- splits on whitespace.

Then apply it to the dataset and print the tokens for one sample message.


In [2]:
import re

NON_ALNUM_RE = re.compile(r'[^a-z0-9]+')


def preprocess_text(text: str, remove_stopwords=False) -> list[str]:
    """Convert text to a list of tokens.
    - Lowercase the text
    - Remove punctuation/non-alphanumeric characters
    - Split into tokens (words)

    TODO: Implement.
    """
        # lowercase
    text = text.lower()
    
    # replace non-alphanumeric characters with space
    text = NON_ALNUM_RE.sub(" ", text)
    
    # split on whitespace
    tokens = text.split()
    
    # optional stopword removal (simple example list)
    if remove_stopwords:
        stopwords = {'the','a','an','is','are','to','and','in','of','for','on','at','with','this','that','it'}
        tokens = [t for t in tokens if t not in stopwords]

    return tokens


all_tokens = [preprocess_text(msg, remove_stopwords=False) for _, msg in data]
print('Sample message:', data[0][1])
print('Tokens (no stop words removed):', all_tokens[0])
print('Tokens (stop words removed):', preprocess_text(data[0][1], remove_stopwords=True))


Sample message: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Tokens (no stop words removed): ['go', 'until', 'jurong', 'point', 'crazy', 'available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'there', 'got', 'amore', 'wat']
Tokens (stop words removed): ['go', 'until', 'jurong', 'point', 'crazy', 'available', 'only', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'there', 'got', 'amore', 'wat']


## Question 3: Stop-word removal

Stop words are common words (e.g., "the", "and") that often carry little signal.

Tasks:
1. Create a `STOP_WORDS` set (you can start with the small starter set below).
2. Update preprocessing so that when `remove_stopwords=True`, you drop tokens in `STOP_WORDS`.
3. Show the token list **with** and **without** stop-word removal for a sample message.


In [3]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
stop_words = set(ENGLISH_STOP_WORDS)

def preprocess_text(text, remove_stopwords=False):
    text = text.lower()
    
    # Remove punctuation and non-alphanumeric characters
    text = re.sub(r'[^a-z0-9]+', ' ', text)

    tokens = text.split()
    
    # If remove_stopwords is True, filter out tokens that are in the stop_words set

    if remove_stopwords:
        tokens = [w for w in tokens if w not in stop_words]
    return tokens

all_tokens = [preprocess_text(msg, remove_stopwords=False) for _, msg in data]
print('Sample message:', data[0][1])
print('Tokens (no stop words removed):', all_tokens[0])
print('Tokens (stop words removed):', preprocess_text(data[0][1], remove_stopwords=True))

Sample message: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Tokens (no stop words removed): ['go', 'until', 'jurong', 'point', 'crazy', 'available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'there', 'got', 'amore', 'wat']
Tokens (stop words removed): ['jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']


## Question 4: Train/test split

We will evaluate on a held-out test set.

Use:
- **train ratio:** 0.75
- **random seed:** 2026

Tasks:
1. Shuffle the dataset with the given seed.
2. Split into `train_data` and `test_data`.
3. Print the sizes of train and test.


In [4]:
import random

TRAIN_RATIO = 0.75
SEED = 2026

# TODO: shuffle + split
random.seed(SEED)
random.shuffle(data)

split_index = int(len(data) * TRAIN_RATIO)
train_data: list[tuple[str, str]] = data[:split_index]
test_data: list[tuple[str, str]] = data[split_index:]

print('Train size:', len(train_data))
print('Test size:', len(test_data))


Train size: 4180
Test size: 1394


## Question 5: Word frequency counts per class

Using the **training set** only:
1. Build word counts for spam and ham (use `Counter`).
2. Compute the total number of tokens in spam and ham.
3. Build the vocabulary `vocab` (set of all unique tokens).


In [5]:
from collections import Counter

# TODO: build spam_word_counts and ham_word_counts
spam_word_counts: Counter[str] = Counter()
ham_word_counts: Counter[str] = Counter()

for label, msg in train_data:
    tokens = preprocess_text(msg, remove_stopwords=True)
    
    if label == 'spam':
        spam_word_counts.update(tokens)
    else:
        ham_word_counts.update(tokens)

# TODO: totals
n_spam_words = sum(spam_word_counts.values())
n_ham_words = sum(ham_word_counts.values())

# TODO: vocabulary
vocab: set[str] = set(spam_word_counts.keys()) | set(ham_word_counts.keys())

print('Vocab size:', len(vocab))
print('Total spam tokens:', n_spam_words)
print('Total ham tokens:', n_ham_words)


Vocab size: 7231
Total spam tokens: 9891
Total ham tokens: 28134


## Question 6: Prior + conditional probabilities (Laplace smoothing)

We treat `spam` as the positive class.

Use **Laplace smoothing** with:
- **alpha = 0.5**

Tasks:
1. Compute prior probabilities `P_spam` and `P_ham` from the training set.
2. Implement `conditional_prob_smooth(word, class_label)` that returns

\[ P(w\mid c) = 
rac{	ext{count}(w,c)+lpha}{N_c + lpha\,|V|} \]

where `N_c` is the total number of tokens in class `c`.


In [6]:
import math

ALPHA = 0.5

# TODO: priors
n_train = len(train_data)

n_spam = sum(1 for label, _ in train_data if label == 'spam')
n_ham = sum(1 for label, _ in train_data if label == 'ham')

P_spam = n_spam / n_train
P_ham = n_ham / n_train

print(f'P(spam) = {P_spam:.4f}')
print(f'P(ham)  = {P_ham:.4f}')


def conditional_prob_smooth(word: str, class_label: str) -> float:
    """Return P(word | class_label) with Laplace smoothing."""
    
    if class_label == 'spam':
        count = spam_word_counts[word]
        N_c = n_spam_words
    else:
        count = ham_word_counts[word]
        N_c = n_ham_words

    return (count + ALPHA) / (N_c + ALPHA * len(vocab))


for w in ['free', 'call', 'txt']:
    print(w, '| spam:', conditional_prob_smooth(w, 'spam'), '| ham:', conditional_prob_smooth(w, 'ham'))


P(spam) = 0.1359
P(ham)  = 0.8641
free | spam: 0.012401436345463295 | ham: 0.001370100316540418
call | spam: 3.701921297153223e-05 | ham: 1.5748279500464574e-05
txt | spam: 0.00899566875208233 | ham: 0.0003307138695097561


# Question 7: Implementing the Naive Bayes Classifier

Using the probabilities we derived, we can now classify a new message as spam or ham.

According to Bayes’ theorem, we compute:

$$
P(\text{spam} \mid \text{message}) \propto P(\text{spam}) \prod_{w \in \text{message}} P(w \mid \text{spam})
$$

$$
P(\text{ham} \mid \text{message}) \propto P(\text{ham}) \prod_{w \in \text{message}} P(w \mid \text{ham})
$$

To avoid numerical underflow from multiplying many small probabilities, we work in the **log space**. This gives us the **log-likelihoods**:

$$
\log P(\text{spam} \mid \text{message}) = \log P(\text{spam}) + \sum_{w \in \text{message}} \log P(w \mid \text{spam})
$$

$$
\log P(\text{ham} \mid \text{message}) = \log P(\text{ham}) + \sum_{w \in \text{message}} \log P(w \mid \text{ham})
$$

We then predict the class with the higher log-probability.

---

Your tasks:

- Implement a function `predict_naive_bayes(message)` that:
  - Preprocesses the input message using your `preprocess_text` function.
  - Computes the **log-probability** of the message being spam and ham using Laplace-smoothed conditional probabilities.
  - Returns the class `"spam"` if the spam log-probability is higher, otherwise `"ham"`.

- Test your function with two example messages:
  - `"Congratulations! You've won a free lottery. Call now to claim $$$"` (likely spam)
  - `"Hey, are we still on for dinner tonight?"` (likely ham)

---

Hint:
Use `math.log()` for computing logarithms, and be sure your probabilities are nonzero (you should use the Laplace-smoothed ones).


In [7]:
def predict_naive_bayes(message):
    """
    Predict whether a given message is "spam" or "ham" using the trained Naive Bayes model.
    Returns the predicted label as a string.
    """
    # Step 1: Preprocess the message
    # TODO: Tokenize and clean the message
    tokens = preprocess_text(message, remove_stopwords=True)

    # Step 2: Initialize log probabilities with prior probabilities
    # TODO: Use math.log on the prior probabilities P_spam and P_ham
    log_prob_spam = math.log(P_spam)
    log_prob_ham = math.log(P_ham)

    # Step 3: Add log conditional probabilities for each word
    for word in tokens:
        if word in vocab:
            # TODO: Add log P(word | spam) and log P(word | ham)
            log_prob_spam = log_prob_spam + math.log(conditional_prob_smooth(word, 'spam'))
            log_prob_ham = log_prob_ham + math.log(conditional_prob_smooth(word, 'ham'))
        # If the word is not in the vocabulary, skip it

    # Step 4: Return the class with the higher log probability
    # TODO: Compare log_prob_spam and log_prob_ham
    if log_prob_spam > log_prob_ham:
        return "spam"
    else:
        return "ham"


tests = [
    "Congratulations! You've won a free lottery. Call now to claim $$$",   # likely spam
    'Hey, are we still on for dinner tonight?',  # likely ham
]
for msg in tests:
    print(msg)
    print('->', predict_naive_bayes(msg))


Congratulations! You've won a free lottery. Call now to claim $$$
-> spam
Hey, are we still on for dinner tonight?
-> ham


## Question 8: Predict on the test set

Tasks:
1. Create `y_true` (true labels) and `y_pred` (predicted labels) for the test set.
2. Print the first 10 `(true, pred)` pairs.


In [8]:
# TODO: predictions

y_true: list[str] = [label for label, _ in test_data]
y_pred: list[str] = [predict_naive_bayes(msg) for _, msg in test_data]

for i in range(10):
    print(y_true[i], '->', y_pred[i])


ham -> ham
ham -> ham
ham -> ham
ham -> ham
ham -> ham
ham -> ham
ham -> ham
ham -> ham
ham -> ham
ham -> ham


## Question 9: Evaluate performance

Compute:
- TP, FP, TN, FN (for spam as the positive class)
- accuracy, precision, recall, F1

Be careful about division by zero.


In [9]:
# TODO: confusion matrix counts
TP = FP = TN = FN = 0

for actual, pred in zip(y_true, y_pred):
    if actual == "spam" and pred == "spam":
        TP += 1
    elif actual == "ham" and pred == "ham":
        TN += 1
    elif actual == "ham" and pred == "spam":
        FP += 1
    elif actual == "spam" and pred == "ham":
        FN += 1

# TODO: metrics
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f'TP={TP} FP={FP} TN={TN} FN={FN}')
print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')


TP=170 FP=9 TN=1206 FN=9
Accuracy:  0.9871
Precision: 0.9497
Recall:    0.9497
F1:        0.9497


## Question 10: Discussion (written)

Write **3–5 sentences each**:

1. Interpret your metrics (precision vs recall tradeoff for a spam filter).
2. Did stop-word removal help in your experiment? Why or why not?
3. Limitations of Naive Bayes for text classification.

**Answer below:**


**Answer:**

1) In the experiment, the spam classifier achieved very good performance. The accuracy was about 0.987, and both precision and recall were around 0.95. High precision means that most messages predicted as spam were actually spam. High recall means the model was able to detect most of the spam messages in the dataset. For a spam filter, it is important to keep a balance between precision and recall so that spam is detected while not incorrectly marking too many normal messages as spam.

2) Yes, stop-word removal likely helped the model because common words like “the”, “and”, and “is” usually do not help identify whether a message is spam or ham. By removing these words, the model focuses more on important words such as “free”, “win”, or “call”, which are more related to spam messages. This reduces unnecessary noise in the data and can make the model simpler. It also reduces the vocabulary size, which can improve efficiency.

3) One limitation of Naive Bayes is that it assumes all words are independent from each other. In real language, words often appear together and influence each other’s meaning. For example, the phrase “free entry” has a stronger meaning together than each word alone. Naive Bayes also ignores the order of words in a sentence. Because of these limitations, more advanced models like neural networks can sometimes perform better for text classification.
